# Your Title Here

**Name(s)**: Xuan Yin

**Website Link**: https://xuan0035.github.io/recipe-rating-prediction/

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

# from dsc80_utils import * # Feel free to uncomment and use this.

## Step 1: Introduction

In [7]:
recipes = pd.read_csv("RAW_recipes.csv")
interactions = pd.read_csv("interactions.csv")
# What recipe characteristics are associated with higher average ratings?
# Do quick recipes receive different average ratings than longer recipes?

## Step 2: Data Cleaning and Exploratory Data Analysis

In [8]:
# Merge recipes and interactions datasets
merged = recipes.merge(
    interactions,
    how="left",
    left_on="id",
    right_on="recipe_id"
)

# Replace invalid ratings (0) with NaN
merged["rating"] = merged["rating"].replace(0, np.nan)

# Compute average rating per recipe
avg_rating = (
    merged
    .groupby("id")["rating"]
    .mean()
    .rename("avg_rating")
)

# Compute number of valid ratings per recipe
n_ratings = (
    merged
    .groupby("id")["rating"]
    .count()  # counts non-NaN ratings
    .rename("n_ratings")
)

# Add average rating and rating counts back to recipes dataset
recipes_with_ratings = (
    recipes
    .merge(avg_rating, left_on="id", right_index=True, how="left")
    .merge(n_ratings, left_on="id", right_index=True, how="left")
)

print(recipes_with_ratings.columns.tolist())
print(recipes_with_ratings.shape)

# Keep recipes with at least 5 ratings
filtered = recipes_with_ratings[
    recipes_with_ratings["n_ratings"] >= 5
]

# Focus on recipes with realistic rating values
filtered = filtered[
    filtered["avg_rating"] >= 3.5
]


# Plot distribution of average ratings
fig = px.histogram(
    filtered,
    x="avg_rating",
    nbins=20,
    title="Distribution of Average Recipe Ratings (Recipes with ≥ 5 Ratings)",
    labels={"avg_rating": "Average Rating"}
)

fig.update_layout(
    xaxis_title="Average Rating",
    yaxis_title="Number of Recipes"
)

fig.show()


# Plot distribution of cooking time
fig = px.histogram(
    filtered,
    x="minutes",
    nbins=40,
    title="Distribution of Recipe Cooking Time",
    labels={"minutes": "Cooking Time (Minutes)"}
)

fig.update_layout(
    xaxis_title="Cooking Time (Minutes)",
    yaxis_title="Number of Recipes"
)

fig.show()


# Plot relationship between cooking time and average rating
fig = px.scatter(
    filtered,
    x="minutes",
    y="avg_rating",
    opacity=0.4,
    title="Relationship Between Cooking Time and Average Rating",
    labels={
        "minutes": "Cooking Time (Minutes)",
        "avg_rating": "Average Rating"
    }
)

fig.show()


# Plot relationship between number of steps and average rating
fig = px.scatter(
    filtered,
    x="n_steps",
    y="avg_rating",
    opacity=0.4,
    title="Relationship Between Number of Steps and Average Rating",
    labels={
        "n_steps": "Number of Steps",
        "avg_rating": "Average Rating"
    }
)

fig.show()


# Group recipes by cooking time quartiles
filtered["time_group"] = pd.qcut(filtered["minutes"], 4)

# Compute average rating and number of recipes per group
aggregate_table = (
    filtered
    .groupby("time_group")
    .agg(
        avg_rating=("avg_rating", "mean"),
        recipe_count=("id", "count")
    )
)

aggregate_table

['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients', 'avg_rating', 'n_ratings']
(83782, 14)


/var/folders/bs/zdqcc7895pl31407h2zjnx540000gn/T/ipykernel_97318/95198864.py:121: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,avg_rating,recipe_count
time_group,,
"(0.999, 20.0]",4.774106,3587
"(20.0, 32.0]",4.756549,1972
"(32.0, 60.0]",4.735662,3142
"(60.0, 20190.0]",4.728009,2378


## Step 3: Assessment of Missingness

In [11]:
# Create a missingness indicator for the description column
filtered["description_missing"] = filtered["description"].isna()


# Observed difference in mean cooking time
obs_minutes_diff = (
    filtered.groupby("description_missing")["minutes"]
    .mean()
    .diff()
    .iloc[-1]
)


# Permutation test for minutes
n = 1000
perm_diffs_minutes = []

for _ in range(n):
    shuffled = filtered.copy()
    shuffled["perm"] = np.random.permutation(shuffled["description_missing"])

    diff = (
        shuffled.groupby("perm")["minutes"]
        .mean()
        .diff()
        .iloc[-1]
    )

    perm_diffs_minutes.append(diff)

perm_diffs_minutes = np.array(perm_diffs_minutes)

p_val_minutes = np.mean(np.abs(perm_diffs_minutes) >= abs(obs_minutes_diff))


# Observed difference in mean number of steps
obs_steps_diff = (
    filtered.groupby("description_missing")["n_steps"]
    .mean()
    .diff()
    .iloc[-1]
)


# Permutation test for number of steps
perm_diffs_steps = []

for _ in range(n):
    shuffled = filtered.copy()
    shuffled["perm"] = np.random.permutation(shuffled["description_missing"])

    diff = (
        shuffled.groupby("perm")["n_steps"]
        .mean()
        .diff()
        .iloc[-1]
    )

    perm_diffs_steps.append(diff)

perm_diffs_steps = np.array(perm_diffs_steps)

p_val_steps = np.mean(np.abs(perm_diffs_steps) >= abs(obs_steps_diff))


obs_minutes_diff, p_val_minutes, obs_steps_diff, p_val_steps

fig = px.box(
    filtered,
    x="description_missing",
    y="minutes",
    title="Cooking Time Distribution by Description Missingness",
    labels={
        "description_missing": "Description Missing",
        "minutes": "Cooking Time (Minutes)"
    }
)

fig.show()

## Step 4: Hypothesis Testing

In [12]:
# Split recipes into quick and long based on median cooking time
median_minutes = filtered["minutes"].median()

quick_recipes = filtered[filtered["minutes"] <= median_minutes]
long_recipes = filtered[filtered["minutes"] > median_minutes]

# Observed difference in mean ratings
observed_diff = (
    quick_recipes["avg_rating"].mean()
    - long_recipes["avg_rating"].mean()
)

# Permutation test
n_simulations = 1000
simulated_diffs = []

for _ in range(n_simulations):

    shuffled = filtered.copy()
    shuffled["shuffled_minutes"] = np.random.permutation(shuffled["minutes"])

    quick = shuffled[shuffled["shuffled_minutes"] <= median_minutes]
    long = shuffled[shuffled["shuffled_minutes"] > median_minutes]

    diff = quick["avg_rating"].mean() - long["avg_rating"].mean()
    simulated_diffs.append(diff)

simulated_diffs = np.array(simulated_diffs)

# Compute p-value
p_value = np.mean(np.abs(simulated_diffs) >= abs(observed_diff))

observed_diff, p_value

(np.float64(0.03551306646729291), np.float64(0.0))

## Step 5: Framing a Prediction Problem

In [ ]:
# TODO

## Step 6: Baseline Model

In [ ]:
# TODO

## Step 7: Final Model

In [ ]:
# TODO

## Step 8: Fairness Analysis

In [ ]:
# TODO